# What to Preserve, What to Discard [Step 02.04]

> **MLCourse - Agentic AI - Agent Patterns**

Every technique so far - truncation, rolling summaries, hierarchies - is a
mechanism. This notebook is about the **policy** those mechanisms enforce.

The question is not "how do I compress?" It is:

> Which facts is my agent **not allowed** to lose, ever?

Answer that first and the mechanism follows. Answer it never - which is the
default - and your compression prompt is quietly making the decision for you.

### What you'll learn

- A four-category retention policy you can actually apply.
- An **append-only fact store**: the cheapest fix for compounding drift.
- A measured comparison: summary alone vs. summary + pinned facts.

### Why it matters

Notebook 02 showed drift: a fact dropped at fold 1 is gone forever. The fix is
not a better summariser - it is to stop routing constraints through the
summariser at all. Extract them once, store them in a list that is never
rewritten, and let the summary handle only the narrative.

### Prerequisites

- [02_rolling_summarization](02_rolling_summarization.ipynb)
- [01_context_engineering/03_trimming_strategies](../01_context_engineering/03_trimming_strategies.ipynb) - priority tiering, the same idea applied to trimming.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


In [2]:
# The shared conversation lives in conversation_data.py next to this notebook,
# because 40 turns pasted at the top of five notebooks would bury the lesson.
from conversation_data import (CONVERSATION, DURABLE_FACTS, EPHEMERAL_MARKERS,
                               PROBE_QUESTIONS, as_text)

print("turns          :", len(CONVERSATION))
print("transcript     : %d approx tokens" % approx_tokens(as_text()))
print("durable facts  :", len(DURABLE_FACTS))
for label, _ in DURABLE_FACTS:
    print("   -", label)

turns          : 42
transcript     : 766 approx tokens
durable facts  : 8
   - product name
   - EU-only hosting
   - budget 12,000 EUR
   - price 49 EUR/shop
   - stack: Postgres + Django
   - no third-party LLM
   - pilot: Radhaus Krueger, March
   - 30-day trial, no free tier


### A grader we will reuse in every notebook of this module


In [ ]:
# The question is never "does the summary read nicely". It is "can the agent
# still answer questions that depend on facts from the start of the thread".

def probe(memory_text: str, label: str, verbose=True):
    """Ask each probe question using ONLY `memory_text` as the agent's memory."""
    SYS = ("You are an assistant continuing a long conversation. The notes below "
           "are ALL you remember of it. Answer from the notes only. If the notes "
           "do not contain the answer, reply exactly: UNKNOWN. Be very brief.")
    hits = []
    for q, expected in PROBE_QUESTIONS:
        out = chat([("system", SYS),
                    ("user", "Your notes:\n%s\n\nQuestion: %s" % (memory_text, q))],
                   temperature=0.0, max_tokens=60)
        a = out.content.strip().lower()
        ok = any(e in a for e in expected)
        hits.append(ok)
        if verbose:
            print("  %s %-52s -> %s" % ("OK  " if ok else "LOST", q[:52],
                                        a.replace("\n", " ")[:60]))
    score = sum(hits) / len(hits)
    print("  %-22s %d/%d  (%.0f%%)  memory size: %d tokens"
          % (label, sum(hits), len(hits), score * 100, approx_tokens(memory_text)))
    return score, hits


### 1. Four categories

Not all conversation content has the same shelf life. This taxonomy covers
essentially everything a chat agent sees:

| Category | Examples | Policy |
|---|---|---|
| **CONSTRAINT** | "EU hosting only", "no third-party LLM", allergies, budget ceilings | **Never discard.** Pin verbatim. |
| **DECISION** | "we chose Django", "price is 49 EUR" | **Never discard**, but may be *superseded* by a later decision. |
| **CONTEXT** | The reasoning behind a decision, options considered | Compress. Keep the conclusion, drop the deliberation. |
| **CHATTER** | Greetings, tangents, advice not acted on | Discard freely. |

The two interesting properties:

- **Constraints and decisions are cheap.** Together they are usually under 150
  tokens for an entire conversation. There is no cost argument for losing them.
- **Decisions can be superseded, constraints usually cannot.** "We said 49 EUR,
  now we say 59 EUR" is an update. "EU hosting only" does not get overwritten by
  a later preference - if it changes, someone says so explicitly.

### 2. Extracting the facts - one call, not one per turn

The extraction runs **once per fold**, over the turns being folded. Not once per
turn (too expensive) and not once over the whole transcript at the end (too late
- the transcript may no longer exist).

Note the output format: strict, tiny, and parseable. We ask for one fact per
line with a category tag. Asking for JSON here is also fine, but line-oriented
output is more robust with small models and costs fewer tokens.

In [4]:
EXTRACT_PROMPT = """From the conversation section below, extract only durable facts.

Output one fact per line in exactly this format:
CONSTRAINT: <fact>
DECISION: <fact>

Rules:
- CONSTRAINT = a hard rule that limits what may be built or done.
- DECISION = a choice that was made, including names, prices, dates and tools.
- Include every name, number and date verbatim.
- Do NOT output reasoning, options considered, small talk, or suggestions that
  were not accepted.
- If the section contains no durable facts, output exactly: NONE

SECTION:
{text}"""


def extract_facts(turns):
    out = chat([("user", EXTRACT_PROMPT.format(text=as_text(turns)))],
               temperature=0.0, max_tokens=220).content.strip()
    facts = []
    for line in out.splitlines():
        line = line.strip().lstrip("-* ").strip()
        if line.upper().startswith(("CONSTRAINT:", "DECISION:")):
            kind, _, body = line.partition(":")
            facts.append((kind.strip().upper(), body.strip()))
    return facts


CHUNK = 8
fact_store = []                      # APPEND ONLY. Never rewritten by an LLM.
for i in range(0, len(CONVERSATION), CHUNK):
    section = CONVERSATION[i:i + CHUNK]
    new = extract_facts(section)
    fact_store.extend(new)
    print("turns %2d-%2d -> %d facts" % (i + 1, i + len(section), len(new)))

print()
for kind, body in fact_store:
    print("%-11s %s" % (kind, body))

turns  1- 8 -> 3 facts


turns  9-16 -> 7 facts


turns 17-24 -> 2 facts


turns 25-32 -> 4 facts


turns 33-40 -> 3 facts


turns 41-42 -> 3 facts

CONSTRAINT  The system must run entirely inside the EU.
DECISION    v1 scope is limited to booking service appointments and tracking repair jobs.
DECISION    The target customers are German bike shops.
DECISION    The product is named Spannerbox.
DECISION    No mobile app will be built for v1.
DECISION    A responsive web app will be used for the counter and workshop.
CONSTRAINT  The budget is 12,000 EUR for the first six months.
DECISION    Managed hosting will be used.
DECISION    No hires will be made.
DECISION    Per-shop flat monthly pricing will be used.
DECISION    Pricing is set at 49 EUR per shop per month.
DECISION    The technology stack is Postgres and Django.
CONSTRAINT  No customer data may ever be sent to a third-party LLM.
DECISION    AI features must run on self-hosted models, or not at all.
DECISION    Use an off-the-shelf component library and resist customising it.
DECISION    Do not build for fax in v1.
DECISION    The first pilot customer i

> **"Append only" is doing real work here.** The fact store is never handed back
> to a model with an instruction to rewrite it. That single property is what
> makes it immune to the compounding drift from notebook 02: nothing can quietly
> reword "12,000 EUR" into "a modest budget" at fold 4.

### 3. Supersession - the one edit you do allow

Decisions change. If the founder later says "actually, 59 EUR", you need the new
value without losing the audit trail.

The rule: **never delete, mark superseded**. Appending an update and marking the
old entry keeps the history while ensuring only the current value reaches the
model.

In [5]:
import re


def supersede(store, new_kind, new_body, key_pattern):
    """Append a new fact and mark any earlier fact matching key_pattern as stale.

    Nothing is deleted and nothing is edited in place - that is the whole point.
    """
    stale = [i for i, (_, body) in enumerate(store)
             if re.search(key_pattern, body, re.I)]
    return store + [(new_kind, new_body)], stale


# Simulate a later turn: the founder revises the price.
fact_store2, stale_idx = supersede(fact_store, "DECISION",
                                   "Price revised to 59 EUR per shop per month",
                                   r"\b49\b")

print("newly appended : DECISION: Price revised to 59 EUR per shop per month")
print("marked stale   :", [fact_store[i][1][:52] for i in stale_idx] or "(nothing matched)")


def render_store(store, stale=()):
    lines = []
    for i, (kind, body) in enumerate(store):
        if i in stale:
            continue
        lines.append("- [%s] %s" % (kind, body))
    return "\n".join(lines)


print("\nWHAT THE MODEL SEES:\n" + render_store(fact_store2, set(stale_idx)))

newly appended : DECISION: Price revised to 59 EUR per shop per month
marked stale   : ['Pricing is set at 49 EUR per shop per month.']

WHAT THE MODEL SEES:
- [CONSTRAINT] The system must run entirely inside the EU.
- [DECISION] v1 scope is limited to booking service appointments and tracking repair jobs.
- [DECISION] The target customers are German bike shops.
- [DECISION] The product is named Spannerbox.
- [DECISION] No mobile app will be built for v1.
- [DECISION] A responsive web app will be used for the counter and workshop.
- [CONSTRAINT] The budget is 12,000 EUR for the first six months.
- [DECISION] Managed hosting will be used.
- [DECISION] No hires will be made.
- [DECISION] Per-shop flat monthly pricing will be used.
- [DECISION] The technology stack is Postgres and Django.
- [CONSTRAINT] No customer data may ever be sent to a third-party LLM.
- [DECISION] AI features must run on self-hosted models, or not at all.
- [DECISION] Use an off-the-shelf component library and resi

### 4. Does pinning actually help? Measure it.

Three memories, same probes:

1. A **lossy summary** on its own (deliberately aggressive, to simulate several
   folds of drift).
2. The **fact store** on its own.
3. **Both** - which is what you would ship.

In [6]:
# A deliberately aggressive summary: this is what a vague prompt produces after
# a few folds. Note it is fluent, plausible, and has lost every number.
lossy = chat([("user",
               "Summarise this conversation in at most 60 words.\n\n" + as_text())],
             temperature=0.0, max_tokens=140).content.strip()

FACTS_TEXT = render_store(fact_store)
BOTH = "PINNED FACTS:\n%s\n\nNARRATIVE SUMMARY:\n%s" % (FACTS_TEXT, lossy)

print("LOSSY SUMMARY (%d tokens):\n%s\n" % (approx_tokens(lossy), lossy))

LOSSY SUMMARY (78 tokens):
Spannerbox is a €12k, EU-only SaaS for German bike shops, using Django/Postgres. V1 focuses on booking and repair tracking, launching in March with a pilot. Key decisions include a €49/month flat fee, 30-day trials, no mobile app, and strict data privacy. The immediate goal is building a functional booking flow for the pilot customer.



In [7]:
print("1) LOSSY SUMMARY ALONE")
s_lossy, _ = probe(lossy, "lossy summary")

1) LOSSY SUMMARY ALONE


  OK   What is the product called?                          -> spannerbox


  LOST Where must the product be hosted, and why?           -> unknown


  OK   What is the monthly price per shop?                  -> €49


  OK   Which database and web framework were chosen?        -> django and postgres.


  LOST Who is the first pilot customer and when do they sta -> unknown


  OK   Is it acceptable to send customer data to a hosted L -> unknown
  lossy summary          4/6  (67%)  memory size: 78 tokens


In [8]:
print("2) FACT STORE ALONE")
s_facts, _ = probe(FACTS_TEXT, "fact store", verbose=False)
print("\n3) FACT STORE + SUMMARY")
s_both, _ = probe(BOTH, "facts + summary", verbose=False)

2) FACT STORE ALONE


  fact store             6/6  (100%)  memory size: 354 tokens

3) FACT STORE + SUMMARY


  facts + summary        6/6  (100%)  memory size: 444 tokens


In [9]:
print("%-26s %9s %9s" % ("memory", "tokens", "recall"))
print("-" * 48)
for name, text, s in (("lossy summary only", lossy, s_lossy),
                      ("fact store only", FACTS_TEXT, s_facts),
                      ("facts + summary", BOTH, s_both)):
    print("%-26s %9d %8.0f%%" % (name, approx_tokens(text), 100 * s))
print("%-26s %9d %8s" % ("full transcript", approx_tokens(as_text()), "(baseline)"))
print()
print("MEASURED, this run, %s:" % GROQ_MODEL)
print("  pinning facts changed recall by %+.0f points, for %d extra tokens."
      % (100 * (s_both - s_lossy), approx_tokens(FACTS_TEXT)))

memory                        tokens    recall
------------------------------------------------
lossy summary only                78       67%
fact store only                  354      100%
facts + summary                  444      100%
full transcript                  766 (baseline)

MEASURED, this run, qwen/qwen3.8-27b:
  pinning facts changed recall by +33 points, for 354 extra tokens.


### Which durable facts did each memory actually retain?


In [ ]:
print("%-34s %8s %8s %8s" % ("durable fact", "summary", "facts", "both"))
print("-" * 62)
for label, markers in DURABLE_FACTS:
    row = []
    for text in (lossy, FACTS_TEXT, BOTH):
        row.append("yes" if any(m in text.lower() for m in markers) else "-")
    print("%-34s %8s %8s %8s" % (label, row[0], row[1], row[2]))


That last table is the real deliverable of this notebook. It shows *which*
facts each strategy keeps, not just a percentage - and it is the table you should
build for your own agent, with your own list of things it is not allowed to
forget.

### 5. Writing the policy down

A retention policy is a short document, and writing it takes an afternoon:

1. **List what must never be lost.** For a support agent: entitlements, prior
   promises, stated constraints. For a coding agent: the target framework, the
   test command, decisions already rejected.
2. **Decide what supersedes what.** Prices change; allergies do not.
3. **Pick the mechanism per category.** Pin constraints; summarise context;
   discard chatter.
4. **Write probe questions**, like `PROBE_QUESTIONS` here, and run them in CI.

Step 4 is the one people skip, and it is the one that turns memory from a hope
into an engineering property.

### 6. Pitfalls

- **Letting the summariser own the constraints.** Anything that passes through a
  rewrite can be reworded away. Route constraints around the summariser.
- **Rewriting the fact store.** The moment an LLM rewrites it, drift is back.
  Append, mark, never edit in place.
- **Deleting superseded facts.** You lose the ability to answer "didn't we say
  49 before?" - and that question gets asked.
- **Extracting per turn.** Expensive and noisy. Extract per fold, over a section.
- **A policy that exists only in someone's head.** Write it down; test it.

### Recap

| Idea | Takeaway |
|---|---|
| Four categories | Constraint, decision, context, chatter - different policies |
| Constraints are cheap | Under ~150 tokens; there is no reason to lose them |
| Append-only store | Immune to compounding drift by construction |
| Supersede, don't delete | Keeps the audit trail, shows only current values |
| Probe questions in CI | Retention becomes a measured property, not a hope |

**Next:** [05_mem0_local](05_mem0_local.ipynb) - a real memory library doing all of
this, running entirely on your own machine.